# P4 Agent 4 · NCS Source Audit

| 항목 | 명세 |
|---|---|
| 목적 | Audit 13,442 NCS units, checksum lineage, duplicates, levels, hierarchy codes, and documented name nulls. |
| 담당 Agent | `P4-A4-NCS` |
| Stage ID | `A4-00-NCS-SOURCE` |
| 입력 | `ncs_mapping/data/processed/ncsUnit.parquet` |
| 처리 | checksum·중복·level·hierarchy·null 감사 |
| 출력 | ncs_units canonical source audit 및 4개 종료 artifact |
| 선행 Gate | `NCS_BASE_READY` |
| 후속 활용 | core AI·IT codeset build |

> Development-only orchestration. Empirical analysis and production promotion are disabled.

In [ ]:
RUN_MODE = "observed-dev"
AGENT_ID = "P4-A4-NCS"
STAGE_ID = "A4-00-NCS-SOURCE"
CONTRACT_VERSION = "2.1.2"
SCHEMA_VERSION = "ncs-base-v1"
DATA_VERSION = "observed-dev-20260806.1"
CRAWL_RELEASE_ID = "CRAWL_20260806_03"
AS_OF_DATE = "2026-08-06"
INPUT_MANIFEST_PATH = "ncs_mapping/data/processed/ncsUnit.parquet"
OUTPUT_ROOT = "ncs_mapping/data/runs/observed-dev/NCS_MAPPING_OBSERVED_20260806_01/A4-00-NCS-SOURCE"
RANDOM_SEED = 20260806
FAIL_ON_GATE = True
EMPIRICAL_ANALYSIS_ALLOWED = False
DATA_PROVENANCE = "OBSERVED_DEVELOPMENT_ONLY"
PROMOTION_ALLOWED = False
DUTY_INPUT_PATH = ""
GOLD_INPUT_PATH = ""
CONTROL_SCHEMA_DIR = ""

In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd

NCS_ROOT = Path.cwd().resolve()
if NCS_ROOT.name != 'ncs_mapping':
    raise RuntimeError('run this notebook with cwd=ncs_mapping')
sys.path.insert(0, str(NCS_ROOT / 'src'))
assert RUN_MODE == 'observed-dev'
assert AGENT_ID == 'P4-A4-NCS' and STAGE_ID.startswith('A4-')
assert RANDOM_SEED == 20260806 and FAIL_ON_GATE is True
assert DATA_PROVENANCE == 'OBSERVED_DEVELOPMENT_ONLY'
assert EMPIRICAL_ANALYSIS_ALLOWED is False and PROMOTION_ALLOWED is False
resolved_duty_input = DUTY_INPUT_PATH or os.environ.get('P4_A2_DUTY_HANDOFF', '')
resolved_gold_input = GOLD_INPUT_PATH or os.environ.get('P4_NCS_GOLD_INPUT', '')
resolved_schema_dir = CONTROL_SCHEMA_DIR or os.environ.get('P4_CONTROL_SCHEMA_DIR', '')

In [ ]:
from p4_ncs.quality.stage_artifacts import sha256_file

ncs_path = NCS_ROOT / 'data/processed/ncsUnit.parquet'
ncs_units = pd.read_parquet(ncs_path)
input_audit = {
    'rows': len(ncs_units),
    'parquetSha256': sha256_file(ncs_path),
    'duplicateNcsUnitCode': int(ncs_units['ncsUnitCode'].duplicated().sum()),
    'levels': sorted(ncs_units['ncsLevel'].dropna().astype(int).unique().tolist()),
    'hierarchyCodeNulls': int(ncs_units[['majorCode','middleCode','minorCode','subCode']].isna().sum().sum()),
    'hierarchyNameNulls': int(ncs_units[['majorName','middleName','minorName','subName']].isna().sum().sum()),
    'rawSha256Distinct': int(ncs_units['rawSha256'].nunique(dropna=True)),
}
assert input_audit['rows'] == 13_442
assert input_audit['duplicateNcsUnitCode'] == 0
assert input_audit['levels'] == list(range(1, 9))
input_audit

In [ ]:
from p4_ncs.workflow.observed import run_stage

stage_manifest = run_stage('A4-00-NCS-SOURCE', root=NCS_ROOT, duty_input_path=resolved_duty_input or None, gold_input_path=resolved_gold_input or None, schema_dir=resolved_schema_dir or None)
stage_manifest

In [ ]:
stage_root = NCS_ROOT / 'data/runs' / RUN_MODE / 'NCS_MAPPING_OBSERVED_20260806_01' / stage_manifest['stageId']
expected_artifacts = {'stage_manifest.json', 'stage_metrics.json', 'stage_quality.csv', 'CHECKSUMS.sha256'}
actual_artifacts = {path.name for path in stage_root.iterdir() if path.is_file()}
assert actual_artifacts == expected_artifacts
termination_summary = {'stageId': stage_manifest['stageId'], 'status': stage_manifest['status'], 'rowCounts': stage_manifest['rowCounts'], 'artifacts': sorted(actual_artifacts)}
termination_summary